# 05 — ProtoSegNet: End-to-End Training
**Stage 5 · Full prototype segmentation pipeline · 3-phase training schedule**

Covers:
1. Architecture inspection — parameter counts, tensor shapes per module
2. Smoke test — forward pass, loss decomposition, gradient flow
3. Training — 3-phase schedule (Phase A → B → C)
4. Training curves — loss components + val Dice per phase
5. Patient-level evaluation — per-class Dice vs. U-Net baseline
6. Key findings for Stage 6 (XAI metrics)

**Training time estimate:** ~100 epochs × 90s/ep ≈ 2.5h CT, 1.3h MRI (MPS, batch=16)

In [ ]:
import sys, os

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import csv
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from pathlib import Path

from src.data.mmwhs_dataset import (
    MMWHSSliceDataset, MMWHSPatientDataset,
    make_dataloaders, LABEL_NAMES, NUM_CLASSES,
)
from src.models.proto_seg_net import ProtoSegNet
from src.models.encoder import HierarchicalEncoder2D
from src.models.prototype_layer import PrototypeProjection, PROTOS_PER_LEVEL
from src.losses.segmentation import SegmentationLoss, compute_class_weights
from src.losses.diversity_loss import ProtoSegLoss
from src.metrics.dice import dice_per_class, mean_foreground_dice

DEVICE     = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR   = 'data/pack/processed_data'
CKPT_DIR   = Path('checkpoints')
RESULT_DIR = Path('results')
COLORS     = plt.get_cmap('tab10', NUM_CLASSES)
FG_NAMES   = [LABEL_NAMES[c] for c in range(1, NUM_CLASSES)]

print(f'Working dir : {os.getcwd()}')
print(f'Device      : {DEVICE}')

---
## 1 · Architecture Inspection

In [ ]:
model = ProtoSegNet(n_classes=8)
total = model.count_parameters()['total']

modules = {
    'Encoder (HierarchicalEncoder2D)': model.encoder,
    'PrototypeLayers (all levels)'   : model.proto_layers,
    'Decoder (dec4–dec1)'            : nn.ModuleList([model.dec4, model.dec3, model.dec2, model.dec1]),
    'Final 1×1 Conv'                 : model.final_conv,
}
print(f'{"Module":<40} {"Parameters":>12}  Share')
print('─' * 65)
for name, mod in modules.items():
    n = sum(p.numel() for p in mod.parameters())
    print(f'{name:<40} {n:>12,}  {100*n/total:5.1f}%')
print('─' * 65)
print(f'{"TOTAL":<40} {total:>12,}  100.0%')
print(f'\nPrototype vectors per level (M_l): {PROTOS_PER_LEVEL}')
print(f'Total prototype vectors: {8 * sum(PROTOS_PER_LEVEL.values())}  (K=8 classes × 11 protos)')

In [ ]:
x_test = torch.randn(1, 1, 256, 256)
model.eval()
with torch.no_grad():
    feat = model.encoder(x_test)
    print('Encoder output (Z_l):')
    for l, z in feat.items():
        print(f'  Level {l}: {tuple(z.shape)}  — {HierarchicalEncoder2D.CHANNELS[l]} ch, stride ×{2**l}')
    print('\nPrototype heatmaps (A_l):')
    for l in [1, 2, 3, 4]:
        A = model._proto_layer(l)(feat[l])
        B, K, M, H, W = A.shape
        print(f'  Level {l}: {tuple(A.shape)}  — (B, K={K}, M={M}, {H}×{W})')
    logits, heatmaps = model(x_test)
    print(f'\nlogits : {tuple(logits.shape)}')
    print(f'heatmaps keys: {list(heatmaps.keys())}')

In [ ]:
def _trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

model.unfreeze_all();                   n_all    = _trainable(model)
model.freeze_prototypes();              n_phaseA = _trainable(model)
model.unfreeze_all()
model.freeze_encoder_and_prototypes();  n_phaseC = _trainable(model)
model.unfreeze_all()

print('Trainable parameters by training phase:')
print(f'  Phase A (backbone + decoder; prototypes frozen) : {n_phaseA:>9,}  ({100*n_phaseA/n_all:.1f}%)')
print(f'  Phase B (all params)                           : {n_all:>9,}  (100.0%)')
print(f'  Phase C (decoder only)                         : {n_phaseC:>9,}  ({100*n_phaseC/n_all:.1f}%)')

---
## 2 · Smoke Test

In [ ]:
model = ProtoSegNet().to(DEVICE)
w = torch.ones(8, device=DEVICE)
seg_loss_test = SegmentationLoss(class_weights=w)
criterion_test = ProtoSegLoss(seg_loss=seg_loss_test, lambda_div=0.01)

x = torch.randn(4, 1, 256, 256, device=DEVICE)
y = torch.randint(0, 8, (4, 256, 256), device=DEVICE)

model.train(); model.unfreeze_all()
logits, heatmaps = model(x)
losses = criterion_test(logits, y, heatmaps)
losses['loss'].backward()

print('Forward + backward: OK')
print(f"  total={losses['loss'].item():.4f}  dice={losses['dice_loss'].item():.4f}  "
      f"CE={losses['ce_loss'].item():.4f}  div={losses['div_loss'].item():.4f}")
for l in [1, 2, 3, 4]:
    g = model.proto_layers[str(l)].prototypes.grad
    status = '✓' if (g is not None and g.abs().sum() > 0) else '✗'
    print(f'  Grad → proto_layers[{l}]: {status}')

---
## 3 · Training

**3-phase schedule:**
| Phase | Epochs | Frozen | Loss |
|---|---|---|---|
| A | 1–20   | Prototypes | 0.5·Dice + 0.5·CE |
| B | 21–80  | Nothing    | 0.5·Dice + 0.5·CE + λ·L_div · projection every 10 ep |
| C | 81–100 | Encoder + Prototypes | 0.5·Dice + 0.5·CE |

**Outputs:** `checkpoints/proto_seg_{ct,mr}.pth` · `checkpoints/projected_prototypes_{ct,mr}.pt` · `results/train_curve_proto_{ct,mr}.csv`

Set `MODALITY` in the config cell and run all cells in this section. Training is blocking (~2.5h CT, ~1.3h MRI).

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
MODALITY           = 'ct'   # ← change to 'mr' for MRI

BATCH_SIZE         = 16
LR                 = 3e-4
WEIGHT_DECAY       = 1e-5
LAMBDA_DIV         = 0.01

PHASE_A_END        = 20
PHASE_B_END        = 80
PHASE_C_END        = 100
MAX_EPOCHS         = PHASE_C_END

VAL_EVERY          = 5
PATIENCE           = 15   # val-check rounds with no improvement before stopping
PROJECTION_INTERVAL = 10  # epochs between prototype projections in Phase B

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

@torch.no_grad()
def validate_slices(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        logits, _ = model(batch['image'].to(DEVICE))
        all_logits.append(logits.cpu())
        all_labels.append(batch['label'])
    model.train()
    return dice_per_class(torch.cat(all_logits), torch.cat(all_labels))


def run_projection(model, modality, save_path):
    print('  [Projection] Building feature bank on CPU…', flush=True)
    t0 = time.time()
    proj_ds = MMWHSSliceDataset(DATA_DIR, modality, 'train', augment=False, preload=True)
    proj_loader = torch.utils.data.DataLoader(proj_ds, batch_size=32, shuffle=False)
    wrapped = [(b['image'], b['label']) for b in proj_loader]

    projector = PrototypeProjection(
        encoder=model.encoder,
        proto_layers=model.proto_layers_dict(),
        device='cpu',
    )
    projector.project(wrapped, save_path=str(save_path))

    ckpt = torch.load(save_path, weights_only=True)
    for level, proto_data in ckpt['proto_state'].items():
        model.proto_layers[str(level)].prototypes.data.copy_(proto_data)
    print(f'  [Projection] Done in {time.time()-t0:.1f}s', flush=True)


def set_phase(model, epoch, optimizer):
    if epoch <= PHASE_A_END:
        model.unfreeze_all(); model.freeze_prototypes(); phase = 'A'
    elif epoch <= PHASE_B_END:
        model.unfreeze_all(); phase = 'B'
    else:
        model.freeze_encoder_and_prototypes(); phase = 'C'
    optimizer.param_groups[0]['params'] = [p for p in model.parameters() if p.requires_grad]
    return phase

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'  Training ProtoSegNet — {MODALITY.upper()}')
print(f'  Device: {DEVICE}  |  Batch: {BATCH_SIZE}  |  Max epochs: {MAX_EPOCHS}')
print(f'  λ_div={LAMBDA_DIV}  |  Projection every {PROJECTION_INTERVAL} epochs (Phase B)')
print(f'{"="*60}\n')

# Data
loaders = make_dataloaders(DATA_DIR, MODALITY, batch_size=BATCH_SIZE)
print(f'  Train: {len(loaders["train"].dataset)} slices  '
      f'Val: {len(loaders["val"].dataset)} slices  '
      f'Test: {len(loaders["test"].dataset)} slices')

# Class weights
weights_path = Path(f'data/class_weights_{MODALITY}.pt')
if weights_path.exists():
    class_weights = torch.load(weights_path, weights_only=True)
    print(f'  Loaded class weights from {weights_path}')
else:
    print('  Computing class weights…')
    class_weights = compute_class_weights(DATA_DIR, MODALITY)
    torch.save(class_weights, weights_path)

# Model + loss
train_model = ProtoSegNet(n_classes=NUM_CLASSES).to(DEVICE)
seg_loss = SegmentationLoss(class_weights=class_weights.to(DEVICE), n_classes=NUM_CLASSES)
criterion = ProtoSegLoss(seg_loss=seg_loss, lambda_div=LAMBDA_DIV)
print(f'  Model params: {train_model.count_parameters()["total"]:,}')

# Optimizer + scheduler
train_model.freeze_prototypes()
optimizer = torch.optim.AdamW(
    [p for p in train_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

# Logging
log_path = RESULT_DIR / f'train_curve_proto_{MODALITY}.csv'
ckpt_path = CKPT_DIR / f'proto_seg_{MODALITY}.pth'
proj_path = CKPT_DIR / f'projected_prototypes_{MODALITY}.pt'

fieldnames = (
    ['epoch', 'phase', 'train_loss', 'train_dice_loss', 'train_ce_loss', 'train_div_loss',
     'val_mean_fg_dice', 'lr', 'epoch_time_s']
    + [f'val_dice_{LABEL_NAMES[c]}' for c in range(1, NUM_CLASSES)]
)
csv_file = open(log_path, 'w', newline='')
writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
writer.writeheader()

best_val_dice, best_epoch, no_improve = 0.0, 0, 0
current_phase = 'A'
print(f'  Phase A: backbone + decoder train; prototypes frozen (epochs 1–{PHASE_A_END})\n')

for epoch in range(1, MAX_EPOCHS + 1):

    # Phase transitions
    new_phase = set_phase(train_model, epoch, optimizer)
    if new_phase != current_phase:
        current_phase = new_phase
        if current_phase == 'B':
            print(f'\n  → Phase B: all params; diversity loss active (epochs {PHASE_A_END+1}–{PHASE_B_END})')
            run_projection(train_model, MODALITY, proj_path)
        elif current_phase == 'C':
            print(f'\n  → Phase C: decoder fine-tuning only (epochs {PHASE_B_END+1}–{PHASE_C_END})')

    # Periodic projection in Phase B
    if (current_phase == 'B'
            and epoch > PHASE_A_END + 1
            and (epoch - PHASE_A_END) % PROJECTION_INTERVAL == 0):
        run_projection(train_model, MODALITY, proj_path)

    # Train epoch
    t0 = time.time()
    train_model.train()
    total_loss = total_dice = total_ce = total_div = 0.0
    n_batches = 0

    for batch in loaders['train']:
        imgs = batch['image'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits, hm = train_model(imgs)

        if current_phase == 'A':
            out = seg_loss(logits, lbls)
            out['div_loss'] = torch.zeros(1, device=DEVICE)
        else:
            out = criterion(logits, lbls, hm)

        out['loss'].backward()
        nn.utils.clip_grad_norm_([p for p in train_model.parameters() if p.requires_grad], 1.0)
        optimizer.step()

        total_loss += out['loss'].item()
        total_dice += out['dice_loss'].item()
        total_ce   += out['ce_loss'].item()
        total_div  += out['div_loss'].item()
        n_batches  += 1

    scheduler.step()
    epoch_time = time.time() - t0
    avg = lambda s: s / n_batches
    current_lr = scheduler.get_last_lr()[0]

    row = {
        'epoch': epoch, 'phase': current_phase,
        'train_loss':      round(avg(total_loss), 5),
        'train_dice_loss': round(avg(total_dice), 5),
        'train_ce_loss':   round(avg(total_ce),   5),
        'train_div_loss':  round(avg(total_div),  5),
        'lr': round(current_lr, 7),
        'epoch_time_s': round(epoch_time, 1),
    }

    # Validation
    if epoch % VAL_EVERY == 0 or epoch == 1:
        val_dice = validate_slices(train_model, loaders['val'])
        val_mean = mean_foreground_dice(val_dice)
        row['val_mean_fg_dice'] = round(val_mean, 5)
        for c in range(1, NUM_CLASSES):
            name = LABEL_NAMES[c]
            v = val_dice.get(name, float('nan'))
            row[f'val_dice_{name}'] = round(v, 4) if v == v else 'nan'

        if val_mean > best_val_dice:
            best_val_dice, best_epoch, no_improve = val_mean, epoch, 0
            torch.save({
                'epoch': epoch, 'model_state': train_model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_val_dice': best_val_dice,
                'class_weights': class_weights, 'lambda_div': LAMBDA_DIV,
            }, ckpt_path)
            flag = ' ← best'
        else:
            no_improve += 1; flag = ''

        print(f'  [{current_phase}] Ep {epoch:3d}/{MAX_EPOCHS} | '
              f'loss={avg(total_loss):.4f} '
              f'(D={avg(total_dice):.4f} CE={avg(total_ce):.4f} div={avg(total_div):.4f}) | '
              f'val={val_mean:.4f}{flag} | lr={current_lr:.2e} | {epoch_time:.1f}s')
    else:
        row['val_mean_fg_dice'] = ''
        for c in range(1, NUM_CLASSES): row[f'val_dice_{LABEL_NAMES[c]}'] = ''
        if epoch % 10 == 0:
            print(f'  [{current_phase}] Ep {epoch:3d}/{MAX_EPOCHS} | '
                  f'loss={avg(total_loss):.4f} | lr={current_lr:.2e} | {epoch_time:.1f}s')

    writer.writerow(row)
    csv_file.flush()

    if current_phase != 'A' and no_improve >= PATIENCE:
        print(f'\n  Early stopping at epoch {epoch} (no improvement for {PATIENCE} val checks)')
        break

csv_file.close()
print(f'\n  Best val mean fg Dice: {best_val_dice:.4f} at epoch {best_epoch}')
print(f'  Checkpoint : {ckpt_path}')
print(f'  Log        : {log_path}')

---
## 4 · Training Curves

Run this section after training completes (or load logs from a previous run).

In [ ]:
logs = {}
for mod in ('ct', 'mr'):
    p = RESULT_DIR / f'train_curve_proto_{mod}.csv'
    if p.exists():
        logs[mod] = pd.read_csv(p)
        print(f'{mod.upper()}: {len(logs[mod])} epochs logged')
    else:
        print(f'{mod.upper()}: log not found — run training first')

PHASE_COLORS = {'A': '#4C72B0', 'B': '#DD8452', 'C': '#55A868'}

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(len(logs), 3, figsize=(17, 4.5 * len(logs)))
if len(logs) == 1: axes = axes[None, :]  # keep 2-D indexing

for row, (mod, log) in enumerate(logs.items()):
    ep = log['epoch']
    max_ep = ep.max()
    for ax in axes[row]:
        ax.axvspan(1,  20,      alpha=0.07, color=PHASE_COLORS['A'])
        ax.axvspan(21, 80,      alpha=0.07, color=PHASE_COLORS['B'])
        ax.axvspan(81, max_ep,  alpha=0.07, color=PHASE_COLORS['C'])

    axes[row, 0].plot(ep, log['train_loss'], lw=1.2, color='steelblue')
    axes[row, 0].set(title=f'{mod.upper()} — Total Loss', ylabel='Loss', xlabel='Epoch')
    axes[row, 0].grid(alpha=0.3)

    axes[row, 1].plot(ep, log['train_dice_loss'], lw=1.2, label='Dice', color='steelblue')
    axes[row, 1].plot(ep, log['train_ce_loss'],   lw=1.2, label='CE',   color='tomato', ls='--')
    axes[row, 1].set(title=f'{mod.upper()} — Dice vs CE', xlabel='Epoch')
    axes[row, 1].legend(fontsize=8); axes[row, 1].grid(alpha=0.3)

    div = log[log['phase'].isin(['B', 'C'])]
    axes[row, 2].plot(div['epoch'], div['train_div_loss'], lw=1.2, color='purple')
    axes[row, 2].set(title=f'{mod.upper()} — Diversity Loss (Phase B+)', ylabel='L_div', xlabel='Epoch')
    axes[row, 2].grid(alpha=0.3)

legend_els = [Patch(color=PHASE_COLORS[p], alpha=0.4, label=f'Phase {p}') for p in 'ABC']
fig.legend(handles=legend_els, loc='upper center', ncol=3, fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(RESULT_DIR / '05_loss_curves.png', dpi=130)
plt.show()

In [ ]:
BASELINES = {'ct': 0.836, 'mr': 0.825}

fig, axes = plt.subplots(1, len(logs), figsize=(8 * len(logs), 5))
if len(logs) == 1: axes = [axes]

for ax, (mod, log) in zip(axes, logs.items()):
    val = log.dropna(subset=['val_mean_fg_dice'])
    max_ep = log['epoch'].max()
    ax.axvspan(1,  20,     alpha=0.07, color=PHASE_COLORS['A'])
    ax.axvspan(21, 80,     alpha=0.07, color=PHASE_COLORS['B'])
    ax.axvspan(81, max_ep, alpha=0.07, color=PHASE_COLORS['C'])

    ax.plot(val['epoch'], val['val_mean_fg_dice'], 'o-', ms=4, lw=1.5,
            color='steelblue', label='ProtoSegNet val')

    baseline = BASELINES.get(mod, 0)
    ax.axhline(baseline,        color='green',  ls='--', lw=1.2, label=f'U-Net {baseline:.3f}')
    ax.axhline(baseline - 0.03, color='orange', ls=':',  lw=1,   label=f'−3% ({baseline-0.03:.3f})')

    best = val.loc[val['val_mean_fg_dice'].idxmax()]
    ax.axvline(best['epoch'], color='steelblue', ls=':', lw=0.8)
    ax.annotate(f"best {best['val_mean_fg_dice']:.3f}\n(ep {int(best['epoch'])})",
                xy=(best['epoch'], best['val_mean_fg_dice']),
                xytext=(best['epoch'] + 3, best['val_mean_fg_dice'] - 0.04),
                fontsize=8, color='steelblue',
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=0.8))

    ax.set(title=f'{mod.upper()} — Validation Dice', xlabel='Epoch',
           ylabel='Mean Foreground Dice', ylim=(0, 1.0))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '05_val_dice.png', dpi=130)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(logs), figsize=(9 * len(logs), 5))
if len(logs) == 1: axes = [axes]

for ax, (mod, log) in zip(axes, logs.items()):
    val = log.dropna(subset=['val_mean_fg_dice'])
    max_ep = log['epoch'].max()
    ax.axvspan(1,  20,     alpha=0.07, color=PHASE_COLORS['A'])
    ax.axvspan(21, 80,     alpha=0.07, color=PHASE_COLORS['B'])
    ax.axvspan(81, max_ep, alpha=0.07, color=PHASE_COLORS['C'])
    for c, name in enumerate(FG_NAMES, start=1):
        col = f'val_dice_{name}'
        if col in val.columns:
            ax.plot(val['epoch'], pd.to_numeric(val[col], errors='coerce'),
                    label=name, color=COLORS(c), lw=1.5, marker='o', ms=3)
    ax.set(title=f'{mod.upper()} — Per-Class Val Dice', xlabel='Epoch', ylabel='Dice', ylim=(0, 1.05))
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '05_per_class_val_dice.png', dpi=130)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(logs), figsize=(8 * len(logs), 4))
if len(logs) == 1: axes = [axes]

for ax, (mod, log) in zip(axes, logs.items()):
    for phase, color in PHASE_COLORS.items():
        s = log[log['phase'] == phase]
        if len(s):
            ax.plot(s['epoch'], s['epoch_time_s'], lw=1, color=color, label=f'Phase {phase}')
    ax.set(title=f'{mod.upper()} — Epoch Time', xlabel='Epoch', ylabel='Seconds')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '05_epoch_time.png', dpi=120)
plt.show()

for mod, log in logs.items():
    print(f'{mod.upper()}  avg: {log["epoch_time_s"].mean():.0f}s/ep  '
          f'total: {log["epoch_time_s"].sum()/3600:.1f}h')

---
## 5 · Patient-Level Evaluation

In [ ]:
@torch.no_grad()
def load_proto_model(modality):
    path = CKPT_DIR / f'proto_seg_{modality}.pth'
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {path} — run training first')
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m = ProtoSegNet().to(DEVICE)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    print(f'{modality.upper()}: epoch {ckpt["epoch"]}, best val Dice {ckpt["best_val_dice"]:.4f}')
    return m

proto_models = {}
for mod in ('ct', 'mr'):
    try:
        proto_models[mod] = load_proto_model(mod)
    except FileNotFoundError as e:
        print(e)

In [ ]:
BASELINE = {
    'ct': {
        'ct_1019': dict(LV=0.855, RV=0.871, LA=0.674, RA=0.886, Myocardium=0.857, Aorta=0.861, PA=0.647),
        'ct_1020': dict(LV=0.894, RV=0.947, LA=0.936, RA=0.892, Myocardium=0.919, Aorta=0.972, PA=0.928),
    },
    'mr': {
        'mr_1019': dict(LV=0.845, RV=0.871, LA=0.949, RA=0.916, Myocardium=0.940, Aorta=0.845, PA=0.738),
        'mr_1020': dict(LV=0.870, RV=0.875, LA=0.946, RA=0.897, Myocardium=0.921, Aorta=0.641, PA=0.728),
    }
}

@torch.no_grad()
def eval_patients(model, modality, split='test'):
    ds = MMWHSPatientDataset(DATA_DIR, modality, split)
    results = {}
    for i in range(len(ds)):
        s = ds[i]
        imgs = s['image'].to(DEVICE)
        logits_list = []
        for si in range(imgs.shape[0]):
            logits, _ = model(imgs[si:si+1])
            logits_list.append(logits.cpu())
        results[s['patient']] = dice_per_class(torch.cat(logits_list), s['label'])
    return results

proto_results = {}
for mod, mdl in proto_models.items():
    print(f'\n{mod.upper()} test patients:')
    proto_results[mod] = eval_patients(mdl, mod)
    for pid, dice in proto_results[mod].items():
        mfg = mean_foreground_dice(dice)
        vals = '  '.join(f'{k[:3]}={v:.3f}' for k, v in dice.items() if k != 'Background' and v == v)
        print(f'  {pid}: mean_fg={mfg:.4f}  [{vals}]')

In [ ]:
if proto_results:
    fig, axes = plt.subplots(len(proto_results), 2, figsize=(16, 5.5 * len(proto_results)))
    if len(proto_results) == 1: axes = axes[None, :]
    x = np.arange(len(FG_NAMES)); w = 0.35

    for row, mod in enumerate([m for m in ('ct', 'mr') if m in proto_results]):
        patients = sorted(proto_results[mod].keys())
        for col, pid in enumerate(patients[:2]):
            ax = axes[row, col]
            proto_vals = [proto_results[mod][pid].get(n, float('nan')) for n in FG_NAMES]
            base_vals  = [BASELINE[mod].get(pid, {}).get(n, float('nan'))  for n in FG_NAMES]
            ax.bar(x - w/2, base_vals,  w, label='U-Net baseline', color='steelblue', alpha=0.7)
            ax.bar(x + w/2, proto_vals, w, label='ProtoSegNet',    color='tomato',    alpha=0.7)
            ax.axhline(0.75, color='green', ls='--', lw=1)
            ax.set_xticks(x); ax.set_xticklabels(FG_NAMES, rotation=30, ha='right')
            ax.set_ylim(0, 1.05)
            pm = mean_foreground_dice(proto_results[mod][pid])
            bm = np.nanmean(base_vals)
            ax.set_title(f'{mod.upper()} · {pid}\nProtoSegNet {pm:.3f} vs U-Net {bm:.3f} (Δ={pm-bm:+.3f})')
            ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULT_DIR / '05_comparison_baseline.png', dpi=130)
    plt.show()

---
## 5b · Qualitative Segmentation Overlays

In [ ]:
@torch.no_grad()
def show_overlays(model, modality, patient_idx=0, n_slices=5):
    ds = MMWHSPatientDataset(DATA_DIR, modality, 'test')
    s = ds[patient_idx]
    imgs, labels, patient = s['image'], s['label'], s['patient']
    fg_slices = [i for i in range(imgs.shape[0]) if labels[i].max() > 0]
    chosen = fg_slices[::max(1, len(fg_slices) // n_slices)][:n_slices]

    fig, axes = plt.subplots(3, len(chosen), figsize=(3.2 * len(chosen), 9))
    fig.suptitle(f'{modality.upper()} · {patient}', fontsize=12)

    for col, si in enumerate(chosen):
        img_np  = imgs[si].squeeze().numpy()
        lbl_np  = labels[si].numpy()
        logits, _ = model(imgs[si].unsqueeze(0).to(DEVICE))
        pred_np = logits.squeeze(0).argmax(0).cpu().numpy()

        def overlay(ax_obj, mask):
            ax_obj.imshow(img_np, cmap='gray')
            rgba = COLORS(mask / NUM_CLASSES)
            rgba[..., 3] = (mask > 0).astype(float) * 0.65
            ax_obj.imshow(rgba); ax_obj.axis('off')

        axes[0, col].imshow(img_np, cmap='gray'); axes[0, col].axis('off')
        axes[0, col].set_title(f's{si}', fontsize=8)
        overlay(axes[1, col], lbl_np)
        overlay(axes[2, col], pred_np)

    for i, label in enumerate(['Image', 'Ground Truth', 'Prediction']):
        axes[i, 0].set_ylabel(label, fontsize=9)

    legend_patches = [mpatches.Patch(color=COLORS(i / NUM_CLASSES), label=LABEL_NAMES[i])
                      for i in range(1, NUM_CLASSES)]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=8,
               bbox_to_anchor=(0.5, -0.03))
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    out = RESULT_DIR / f'05_overlay_{modality}_{patient}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {out}')

for mod, mdl in proto_models.items():
    show_overlays(mdl, mod, patient_idx=0)
    show_overlays(mdl, mod, patient_idx=1)

---
## 5c · Prototype Heatmap Overlay (preview for Stage 6)

In [ ]:
@torch.no_grad()
def show_heatmaps(model, modality, patient_idx=0, slice_idx=None, level=3, class_id=1):
    ds = MMWHSPatientDataset(DATA_DIR, modality, 'test')
    s = ds[patient_idx]
    imgs, labels = s['image'], s['label']

    if slice_idx is None:
        cands = [i for i in range(imgs.shape[0]) if (labels[i] == class_id).any()]
        slice_idx = cands[len(cands) // 2] if cands else 0

    img_np = imgs[slice_idx].squeeze().numpy()
    _, heatmaps = model(imgs[slice_idx].unsqueeze(0).to(DEVICE))
    A_class = heatmaps[level][0, class_id]  # (M, H_l, W_l)
    M = A_class.shape[0]

    fig, axes = plt.subplots(1, M + 1, figsize=(3.5 * (M + 1), 3.5))
    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title(f'Input (s{slice_idx})', fontsize=9); axes[0].axis('off')

    for m in range(M):
        h = A_class[m].cpu().numpy()
        h_up = torch.nn.functional.interpolate(
            torch.tensor(h)[None, None], size=(256, 256), mode='bilinear', align_corners=False
        ).squeeze().numpy()
        axes[m+1].imshow(img_np, cmap='gray')
        axes[m+1].imshow(h_up, cmap='hot', alpha=0.5, vmin=h_up.min(), vmax=h_up.max())
        axes[m+1].set_title(f'L{level} · {LABEL_NAMES[class_id]} · p{m}', fontsize=8)
        axes[m+1].axis('off')

    plt.suptitle(f'{modality.upper()} — prototype heatmaps  '
                 f'(class={LABEL_NAMES[class_id]}, level={level})', fontsize=10)
    plt.tight_layout()
    out = RESULT_DIR / f'05_heatmap_{modality}_l{level}_{LABEL_NAMES[class_id]}.png'
    plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
    print(f'Saved: {out}')

if proto_models:
    mod = list(proto_models.keys())[0]
    show_heatmaps(proto_models[mod], mod, level=3, class_id=1)  # LV
    show_heatmaps(proto_models[mod], mod, level=2, class_id=6)  # Aorta
    show_heatmaps(proto_models[mod], mod, level=1, class_id=7)  # PA

---
## 6 · Key Findings for Stage 6 (XAI Metrics)

In [ ]:
print(f'{"Patient":<12} {"Model":<16} {"Mean Fg":>8}  ' +
      '  '.join(f'{n[:3]:>5}' for n in FG_NAMES))
print('─' * 90)

for mod in ('ct', 'mr'):
    if mod not in proto_results: continue
    print(f'--- {mod.upper()} ---')
    for pid in sorted(BASELINE[mod]):
        b  = BASELINE[mod][pid]
        bm = np.nanmean(list(b.values()))
        bv = '  '.join(f'{b.get(n, float("nan")):>5.3f}' for n in FG_NAMES)
        print(f'{pid:<12} {"U-Net":<16} {bm:>8.3f}  {bv}')
        if pid in proto_results.get(mod, {}):
            pr = proto_results[mod][pid]
            pm = mean_foreground_dice(pr)
            pv = '  '.join(f'{pr.get(n, float("nan")):>5.3f}' for n in FG_NAMES)
            flag = '✅' if pm - bm >= -0.03 else '⚠️'
            print(f'{"":<12} {"ProtoSegNet":<16} {pm:>8.3f}  {pv}  Δ={pm-bm:+.3f} {flag}')
        print()

### Notes for Stage 6

| Check | Target | Status |
|---|---|---|
| ProtoSegNet Dice within 3% of U-Net | ≥0.81 CT / ≥0.80 MRI | *fill after training* |
| Heatmaps non-trivial | visual check | *see Section 5c* |
| Diversity loss converged (Phase B) | L_div decreasing | *see Section 4 curves* |
| Early stopping epoch | < 100 | *fill after training* |

**XAI metrics to implement next (Stage 6):**
- **AP (Activation Precision):** `heatmaps_dict` already returned by `model(x)` ← ready
- **IDS:** per-slice re-inference with progressive pixel zeroing
- **Faithfulness:** single-pixel perturbation → Pearson correlation
- **Stability:** Gaussian noise perturbations on input